In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = 'course_lead_scoring.csv'

In [3]:
df = pd.read_csv(data)

df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [4]:
df.isna().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [5]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
df.nunique()

lead_source                    5
industry                       7
number_of_courses_viewed      10
annual_income               1267
employment_status              4
location                       7
interaction_count             12
lead_score                   101
converted                      2
dtype: int64

In [7]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

In [8]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [9]:
df[numerical].nunique()

number_of_courses_viewed      10
annual_income               1268
interaction_count             12
lead_score                   101
dtype: int64

In [10]:
df[categorical].nunique()

lead_source          6
industry             8
employment_status    5
location             8
dtype: int64

In [11]:
df.industry.mode()

0    retail
Name: industry, dtype: str

To create a correlation matrix for the numerical features, let's create a separate DataFrame.


In [12]:
corr_matrix = df[numerical].corr()
corr_matrix

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
train_test_split?

Signature:
train_test_split(
    *arrays,
    test_size=None,
    train_size=None,
    random_state=None,
    shuffle=True,
    stratify=None,
)
Docstring:
Split arrays or matrices into random train and test subsets.

Quick utility that wraps input validation,
``next(ShuffleSplit().split(X, y))``, and application to input data
into a single call for splitting (and optionally subsampling) data into a
one-liner.

Read more in the :ref:`User Guide <cross_validation>`.

Parameters
----------
*arrays : sequence of indexables with same length / shape[0]
    Allowed inputs are lists, numpy arrays, scipy-sparse
    matrices or pandas dataframes.

test_size : float or int, default=None
    If float, should be between 0.0 and 1.0 and represent the proportion
    of the dataset to include in the test split. If int, represents the
    absolute number of test samples. If None, the value is set to the
    complement of the train size. If ``train_size`` is also None, it will
    be set to 0.25.

trai

In [15]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

len(df_train), len(df_val), len(df_test), len(df_full_train)

(876, 293, 293, 1169)

In [16]:
df_train = df_train.reset_index(drop=True)
df_val =df_val.reset_index(drop=True)
df_test= df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [17]:
from sklearn.metrics import mutual_info_score

def mut_info_converted(series):
    return round(mutual_info_score(series, y_train), 2)

mi = df_train[categorical].apply(mut_info_converted)
mi.sort_values(ascending=False)

lead_source          0.04
industry             0.01
employment_status    0.01
location             0.00
dtype: float64

### time to use the logistic regression

```python

def sigmoid:
    return 1 / (1 + np.exp(-z))

def logistic_regression:
    score = w0

    for j in range(len(xi)):
        score = score + xi[j] * w[j]

    result = sigmoid(score)
    return result
```

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [19]:
model = LogisticRegression(solver = 'liblinear', C=1.0, max_iter=1000, random_state = 42)
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

In [20]:
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)


model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [21]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)


original_accuracy = (y_val == converted_decision).mean()

round((y_val == converted_decision).mean(),2)

np.float64(0.7)

### Now, let's do the feature elimination part. 
We have as our initial accuracy of our model 70%. By removing each feature separately, we will compare the accuracy and conclude what shall we remove.

industry, employment_status, lead_score?

In [34]:
train_dicts = df_train[categorical + numerical].drop(columns=['industry']).to_dict(orient='records')
val_dicts = df_val[categorical + numerical].drop(columns=['industry']).to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)


model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)

abs(original_accuracy - (y_val == converted_decision).mean()) # Excluded 'industry' feature

np.float64(0.0)

In [35]:
train_dicts = df_train[categorical + numerical].drop(columns=['employment_status']).to_dict(orient='records')
val_dicts = df_val[categorical + numerical].drop(columns=['employment_status']).to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)


model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)

abs(original_accuracy - (y_val == converted_decision).mean()) # Excluded 'employment_status' feature

np.float64(0.0034129692832763903)

In [36]:
train_dicts = df_train[categorical + numerical].drop(columns=['lead_score']).to_dict(orient='records')
val_dicts = df_val[categorical + numerical].drop(columns=['lead_score']).to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)


model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)

abs(original_accuracy - (y_val == converted_decision).mean()) # Excluded 'lead_score' feature

np.float64(0.0068259385665528916)

We see, that the least impactful feature is industry, so in real scenarios, it is better to avoid using that feature in the training dataset.

In [46]:
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

for regul in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', C=regul, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    accuracy = (y_val == converted_decision).mean()
    print(y_pred[:5])

    print(regul, accuracy, sep='\n')
    print()

[0.61192389 0.78000585 0.5269044  0.46836272 0.56737315]
0.01
0.6996587030716723

[0.61193591 0.79790031 0.52976761 0.47097629 0.57026821]
0.1
0.6996587030716723

[0.61192163 0.79982617 0.53021344 0.47131479 0.57066131]
1
0.6996587030716723

[0.61192003 0.80002017 0.53025988 0.47134953 0.57070183]
10
0.6996587030716723

[0.61191986 0.80003958 0.53026453 0.47135301 0.57070588]
100
0.6996587030716723



In [44]:
print(y_val.mean())
print(converted_decision.mean())

0.5563139931740614
0.8020477815699659


In [45]:
print(y_pred[:5])

[0.61191986 0.80003958 0.53026453 0.47135301 0.57070588]


The features in this dataset have very weak predictive signal (MI scores near 0, correlations near 0). Because of that, the model's weights stay small regardless of how strong or weak the regularization is. Different C values shift the raw probabilities slightly, but never enough to push any sample across the 0.5 decision boundary — so the binary classifications end up identical for all five models.